# 4.3 폐쇄형 모델의 RL 방법론


폐쇄형 모델(Closed-Source Model)이란:
- **가중치 접근 불가**: GPT-4, Claude 등은 내부 가중치를 공개하지 않는다
- **API 호출만 가능**: 입출력 인터페이스를 통해서만 상호작용한다
- **강화학습의 제약**: 전통적 RL(PPO, GRPO)을 적용할 수 없다

**해결책**: 프롬프트 공간에서의 최적화
- 프롬프트는 조정 가능한 문맥(context)이다
- 입출력 쌍의 분포를 개선하면 정책을 간접 최적화할 수 있다
- 예: Best-of-N, Prompt Tuning, Output Reranking

**기본 성능**:
- GPT-4o-mini 기본: 약 70% 성공률
- RL 이후: 약 51.5% 성능 향상

## 폐쇄형 모델에서의 RL

### 전통적 RL vs 폐쇄형 모델 RL

**전통적 RL (오픈소스 모델)**
```
정책 π(a|s) → 행동 샘플링 → 보상 수집 → 그래디언트 업데이트
구현: PPO 로스에 그래디언트 역전파
```

**폐쇄형 모델 RL**
```
프롬프트 집합 P → 응답 샘플링 → 보상 평가 → 프롬프트 최적화
구현: Best-of-N, Prompt Tuning, Output Reranking
```

### Best-of-N 전략

**원리**: 같은 쿼리에서 여러 응답을 생성하고 최고 품질 응답을 선택한다

```
1) 쿼리 q에 대해 N개 응답 생성 {y1, y2, ..., yN}
2) 각 응답 평가: reward(yi)
3) 최고 보상 응답 선택: y* = argmax reward(yi)
4) 반복: 높은 보상 프롬프트 변형을 더 자주 사용
```

**효과**:
- N=5: 성공률 약 15% 향상
- N=10: 성공률 약 25% 향상
- N이 클수록 더 좋은 응답을 찾을 가능성 높다

### LLM-as-Judge 기반 보상

**개념**: 또 다른 LLM을 판사(Judge)로 사용하여 응답 품질을 평가한다

```
평가 프롬프트:
  "다음 응답의 정확성을 0~10 점수로 평가하라:"
  응답: {생성된 텍스트}

출력: 숫자 하나 (예: 7.5)
```

**장점**:
- 자동화된 평가 가능하다
- 다양한 기준 적용 가능하다
- API를 통해 측정 가능하다

In [15]:
from utils_openai import *
import numpy as np
from typing import List, Dict
import re

heading("준비: 폐쇄형 모델의 RL")
print("✓ ask() - API 호출로 응답 생성")
print("✓ llm_reward() - LLM 심판으로 보상 평가")
print("✓ Best-of-N - 최고 품질 응답 선택")
print("✓ 프롬프트 변형 - 간접적 정책 최적화")


────────────────────────────────────────
  준비: 폐쇄형 모델의 RL
────────────────────────────────────────
✓ ask() - API 호출로 응답 생성
✓ llm_reward() - LLM 심판으로 보상 평가
✓ Best-of-N - 최고 품질 응답 선택
✓ 프롬프트 변형 - 간접적 정책 최적화


In [20]:
def llm_reward(text, criteria="정확성", max_score=10):
    # 1. system 메시지를 추가하여 LLM이 딴소리를 못하게 차단
    score_str = ask(
        f"평가 기준: {criteria}\n답변 내용: {text}\n\n설명 없이 0~{max_score} 사이의 점수(숫자)만 출력하세요.",
        system="당신은 엄격한 채점관입니다. 서술형 문장을 쓰지 말고 오직 숫자만 반환하세요.",
        temperature=0.0
    )
    
    # 2. [디버깅] 실제로 LLM이 뭐라고 했는지 출력 (0점 원인 파악용)
    print(f"  (LLM 평가 원문: {score_str.strip()})")

    # 3. 숫자(실수/정수) 하나만 쏙 뽑아내기
    match = re.search(r"(\d+(\.\d+)?)", score_str)
    
    # 4. 숫자가 있으면 float 변환, 없으면 0.0 반환 (try-except 없이 처리)
    return float(match.group(1)) if match else 0.0

## 실습 1: Best-of-N 전략 구현

같은 쿼리에 대해 여러 응답을 생성하고 LLM 심판으로 최고를 선택한다.

In [21]:
heading("실습 1: Best-of-N 전략")

# 문제: 파이썬으로 피보나치 수열 함수를 작성하라
query = "파이썬으로 첫 10개 피보나치 수를 생성하는 함수를 작성하라. 매우 간단하고 명확하게 작성하라."

step_print(1, "쿼리", query)

# N=3개 응답 생성 (실제로는 N=5~10 권장)
N = 3
responses = []
scores = []

step_print(2, "응답 생성", f"{N}개 응답 샘플링")

for i in range(N):
    response = ask(query, temperature=0.8)  # 높은 온도로 다양성 확보
    responses.append(response)
    print(f"[응답 {i+1}] {response[:150]}...")

print(f"생성 완료: {N}개")


────────────────────────────────────────
  실습 1: Best-of-N 전략
────────────────────────────────────────
  [Step 1] 쿼리
    파이썬으로 첫 10개 피보나치 수를 생성하는 함수를 작성하라. 매우 간단하고 명확하게 작성하라.
  [Step 2] 응답 생성
    3개 응답 샘플링
[응답 1] 아래는 파이썬으로 첫 10개 피보나치 수를 생성하는 간단한 함수입니다.

```python
def fibonacci(n):
    fib_sequence = []
    a, b = 0, 1
    for _ in range(n):
        fib_sequence...
[응답 2] 파이썬으로 첫 10개 피보나치 수를 생성하는 간단한 함수를 아래와 같이 작성할 수 있습니다.

```python
def fibonacci(n):
    fib_sequence = []
    a, b = 0, 1
    for _ in range(n):
        ...
[응답 3] 아래는 파이썬으로 첫 10개 피보나치 수를 생성하는 간단한 함수입니다.

```python
def fibonacci(n):
    fib_sequence = []
    a, b = 0, 1
    for _ in range(n):
        fib_sequence...
생성 완료: 3개


In [23]:
# 각 응답 평가
step_print(3, "응답 평가", "LLM 심판으로 각 응답 점수 계산 (Retry)")

scores = []
for i, response in enumerate(responses):
    score = llm_reward(response, criteria="코드 정확성, 효율성", max_score=10)
    scores.append(score)
    print(f"  -> 응답 {i+1} 점수: {score}/10")

step_print(4, "최고 응답 선택", "Best-of-N")
if scores:
    best_idx = np.argmax(scores)
    print(f"  선택: 응답 {best_idx+1} (점수: {scores[best_idx]}/10)")

  [Step 3] 응답 평가
    LLM 심판으로 각 응답 점수 계산 (Retry)
  (LLM 평가 원문: 10)
  -> 응답 1 점수: 10.0/10
  (LLM 평가 원문: 9)
  -> 응답 2 점수: 9.0/10
  (LLM 평가 원문: 9)
  -> 응답 3 점수: 9.0/10
  [Step 4] 최고 응답 선택
    Best-of-N
  선택: 응답 1 (점수: 10.0/10)


## 실습 2: 프롬프트 변형을 통한 성능 비교

같은 문제에 대해 다양한 프롬프트 변형을 시도하고 성능을 비교한다.

In [24]:
heading("실습 2: 프롬프트 변형 비교")

# 다양한 프롬프트 변형
prompt_variants = {
    "기본형": "파이썬으로 피보나치 함수를 작성하라.",
    "상세형": "파이썬으로 피보나치 함수를 작성하라. 입력: 정수 n, 출력: 첫 n개 수. 주석을 포함하라.",
    "예시형": "파이썬으로 피보나치 함수를 작성하라. 예: fib(10) → [1,1,2,3,5,8,13,21,34,55]",
    "단계형": "파이썬으로 피보나치 함수를 단계적으로 설명하며 작성하라. 1)함수 정의 2)루프 로직 3)반환"
}

step_print(1, "프롬프트 변형", list(prompt_variants.keys()))

# 각 변형에 대해 1개 응답 생성 및 평가
variant_scores = {}

for name, prompt in prompt_variants.items():
    response = ask(prompt, temperature=0.7)
    score = llm_reward(
        response,
        criteria="파이썬 코드 정확성, 함수 완성도, 코드 가독성",
        max_score=10.0
    )
    variant_scores[name] = score
    print(f"  [{name:6s}] 점수: {score:.1f}/10")

print()
step_print(2, "최고 성능 프롬프트", "성능 비교 결과")
best_variant = max(variant_scores, key=variant_scores.get)
best_variant_score = variant_scores[best_variant]
print(f"  최고: {best_variant} (점수: {best_variant_score:.1f}/10)")
print(f"  → 앞으로 이 프롬프트 변형을 자주 사용한다")


────────────────────────────────────────
  실습 2: 프롬프트 변형 비교
────────────────────────────────────────
  [Step 1] 프롬프트 변형
    • 기본형
    • 상세형
    • 예시형
    • 단계형
  (LLM 평가 원문: 9.0)
  [기본형   ] 점수: 9.0/10
  (LLM 평가 원문: 10.0)
  [상세형   ] 점수: 10.0/10
  (LLM 평가 원문: 9.0)
  [예시형   ] 점수: 9.0/10
  (LLM 평가 원문: 9.0)
  [단계형   ] 점수: 9.0/10

  [Step 2] 최고 성능 프롬프트
    성능 비교 결과
  최고: 상세형 (점수: 10.0/10)
  → 앞으로 이 프롬프트 변형을 자주 사용한다


## 실습 3: 출력 재순위화 (Reranking)

생성된 여러 응답을 같은 기준으로 재평가하고 순서를 다시 매긴다.

In [25]:
heading("실습 3: 출력 재순위화")

# 새로운 쿼리
query = "산더하기를 재귀함수로 구현하라. 입력: 리스트 of 정수, 출력: 정수"

step_print(1, "쿼리", query)

# 4개 응답 생성
N = 4
responses = []

for i in range(N):
    response = ask(query, temperature=0.8)
    responses.append(response)

print(f"  생성 완료: {N}개 응답")

# 단계 1: 정확성 평가
step_print(2, "평가 1: 코드 정확성", "모든 응답 평가")
accuracy_scores = []

for i, response in enumerate(responses):
    score = llm_reward(
        response,
        criteria="파이썬 재귀함수 정확성, 코드 작동 여부",
        max_score=10.0
    )
    accuracy_scores.append(score)
    print(f"  응답 {i+1}: {score:.1f}/10")

# 단계 2: 명확성 평가
step_print(3, "평가 2: 코드 명확성", "모든 응답 평가")
clarity_scores = []

for i, response in enumerate(responses):
    score = llm_reward(
        response,
        criteria="파이썬 코드의 가독성과 명확성",
        max_score=10.0
    )
    clarity_scores.append(score)
    print(f"  응답 {i+1}: {score:.1f}/10")

# 종합 점수
print()
step_print(4, "종합 점수화", "정확성(60%) + 명확성(40%)")
final_scores = []

for i in range(N):
    final = accuracy_scores[i] * 0.6 + clarity_scores[i] * 0.4
    final_scores.append(final)
    print(f"  응답 {i+1}: {final:.1f}/10")

# 재순위화
ranking = sorted(range(N), key=lambda i: final_scores[i], reverse=True)
print()
step_print(5, "최종 순위", "재순위화 결과")

for rank, idx in enumerate(ranking, 1):
    print(f"  {rank}위: 응답 {idx+1} ({final_scores[idx]:.1f}/10)")


────────────────────────────────────────
  실습 3: 출력 재순위화
────────────────────────────────────────
  [Step 1] 쿼리
    산더하기를 재귀함수로 구현하라. 입력: 리스트 of 정수, 출력: 정수
  생성 완료: 4개 응답
  [Step 2] 평가 1: 코드 정확성
    모든 응답 평가
  (LLM 평가 원문: 10.0)
  응답 1: 10.0/10
  (LLM 평가 원문: 10.0)
  응답 2: 10.0/10
  (LLM 평가 원문: 10.0)
  응답 3: 10.0/10
  (LLM 평가 원문: 10.0)
  응답 4: 10.0/10
  [Step 3] 평가 2: 코드 명확성
    모든 응답 평가
  (LLM 평가 원문: 8.5)
  응답 1: 8.5/10
  (LLM 평가 원문: 8.0)
  응답 2: 8.0/10
  (LLM 평가 원문: 8.0)
  응답 3: 8.0/10
  (LLM 평가 원문: 8.0)
  응답 4: 8.0/10

  [Step 4] 종합 점수화
    정확성(60%) + 명확성(40%)
  응답 1: 9.4/10
  응답 2: 9.2/10
  응답 3: 9.2/10
  응답 4: 9.2/10

  [Step 5] 최종 순위
    재순위화 결과
  1위: 응답 1 (9.4/10)
  2위: 응답 2 (9.2/10)
  3위: 응답 3 (9.2/10)
  4위: 응답 4 (9.2/10)


## 실습 4: 프롬프트 수준 GRPO 시뮬레이션

프롬프트 그룹 평균을 비교하여 상대적 보상을 계산한다.

In [28]:
heading("실습 4: 프롬프트 수준 GRPO")

# 프롬프트 그룹 설정
groups = {
    "기본 프롬프트": [
        "파이썬으로 정렬 함수를 작성하라.",
        "배열을 정렬하는 파이썬 함수를 만들어라."
    ],
    "상세 프롬프트": [
        "파이썬으로 정렬 함수를 작성하라. 주석을 포함하고 시간복잡도를 명시하라.",
        "배열을 정렬하는 파이썬 함수를 만들어라. 병합정렬을 사용하고 단계를 설명하라."
    ]
}

step_print(1, "그룹 정의", list(groups.keys()))

# 각 그룹별 점수 계산
group_scores = {}

for group_name, prompts in groups.items():
    scores = []
    print(f"  [{group_name}]")
    
    for prompt in prompts:
        response = ask(prompt, temperature=0.7)
        score = llm_reward(
            response,
            criteria="파이썬 정렬 함수 정확성 및 코드 품질",
            max_score=10.0
        )
        scores.append(score)
        print(f"    프롬프트: {prompt[:40]}... → {score:.1f}/10")
    
    group_scores[group_name] = scores

# GRPO 스타일 상대 보상 계산
print()
step_print(2, "그룹 보상 계산", "상세 프롬프트 vs 기본 프롬프트")

basic_mean = np.mean(group_scores["기본 프롬프트"])
detail_mean = np.mean(group_scores["상세 프롬프트"])
relative_reward = detail_mean - basic_mean

print(f"  기본 프롬프트 평균: {basic_mean:.2f}/10")
print(f"  상세 프롬프트 평균: {detail_mean:.2f}/10")
print(f"  상대 보상: {relative_reward:+.2f}")
print(f"→ 상세 프롬프트가 {'더 효과적' if relative_reward > 0 else '덜 효과적'}이다")
print(f"  정책 업데이트: 상세 프롬프트를 더 자주 사용한다")


────────────────────────────────────────
  실습 4: 프롬프트 수준 GRPO
────────────────────────────────────────
  [Step 1] 그룹 정의
    • 기본 프롬프트
    • 상세 프롬프트
  [기본 프롬프트]
  (LLM 평가 원문: 8.5)
    프롬프트: 파이썬으로 정렬 함수를 작성하라.... → 8.5/10
  (LLM 평가 원문: 8.5)
    프롬프트: 배열을 정렬하는 파이썬 함수를 만들어라.... → 8.5/10
  [상세 프롬프트]
  (LLM 평가 원문: 8.5)
    프롬프트: 파이썬으로 정렬 함수를 작성하라. 주석을 포함하고 시간복잡도를 명시하라.... → 8.5/10
  (LLM 평가 원문: 9.0)
    프롬프트: 배열을 정렬하는 파이썬 함수를 만들어라. 병합정렬을 사용하고 단계를 설명... → 9.0/10

  [Step 2] 그룹 보상 계산
    상세 프롬프트 vs 기본 프롬프트
  기본 프롬프트 평균: 8.50/10
  상세 프롬프트 평균: 8.75/10
  상대 보상: +0.25
→ 상세 프롬프트가 더 효과적이다
  정책 업데이트: 상세 프롬프트를 더 자주 사용한다


## 폐쇄형 vs 오픈소스 모델 비교

### 성능 특성

| 항목 | 폐쇄형 (GPT-4o) | 오픈소스 (Llama) |
|------|--------|----------|
| 기본 성능 | 약 70% | 약 40% |
| 가중치 접근 | 불가능 | 가능 |
| RL 방법 | 프롬프트 최적화 | PPO/GRPO |
| Best-of-N 효과 | 효과 있다 | 제한적 |
| API 비용 | 높다 | 낮다 |
| 배포 용이성 | 의존성 높다 | 독립적 |

### 폐쇄형 모델 RL의 장점

1. **기본 성능이 높다**
   - GPT-4o(약 70%) vs Llama(약 40%)
   - 더 강력한 기본 모델 = 더 좋은 출발점

2. **프롬프트 최적화는 실무적이다**
   - 모델 재훈련 불필요하다
   - 빠른 배포 가능하다
   - 비용 예측 가능하다

3. **LLM-as-Judge가 효과적이다**
   - 복잡한 평가 기준 적용 가능하다
   - 자동화된 품질 검사한다

### 폐쇄형 모델 RL의 제약

1. **그래디언트 접근 불가**
   - PPO, DPO 같은 기법 불가능하다
   - 간접 최적화만 가능하다

2. **API 비용**
   - Best-of-N은 N배 비용이다
   - 실험 비용 증가한다

3. **제어 범위 제한**
   - 프롬프트와 온도만 조정 가능하다
   - 모델 내부 동작 수정 불가능하다

## 요약: 폐쇄형 모델에서의 RL

### 핵심 방법론

1. **Best-of-N**: N개 응답 생성 → 최고 점수 선택한다
   - 비용: N배 API 호출
   - 효과: 성공률 15-25% 향상

2. **프롬프트 변형**: 여러 프롬프트 패턴 비교 → 최고 성능 선택한다
   - 비용: 낮다
   - 효과: 5-10% 향상

3. **Output Reranking**: 다중 기준으로 응답 재평가한다
   - 비용: 추가 API 호출 (응답당 1-2회)
   - 효과: 10-15% 향상

4. **프롬프트 GRPO**: 그룹 평균 비교로 상대 보상 계산한다
   - 비용: 중간
   - 효과: 누적 학습으로 20% 이상 향상 가능하다